# 🎯 5. Generators

Generators are functions that produce values **lazily** — one at a time, on demand.
They're the backbone of Python's memory-efficient iteration.

| ☕ Java | 🐍 Python |
|---------|----------|
| `Iterator<T>` interface (class + `hasNext`/`next`) | `yield` keyword — one line |
| `Stream.iterate()` / `Stream.generate()` | Generator functions / expressions |
| Streams are single-use | Generators are single-use too |

In [11]:
import sys
from typing import Generator
from itertools import islice, chain, takewhile, count, groupby

## 5.1 Generator Functions — `yield`

A function with `yield` becomes a **generator function**.
When called, it returns a **generator object** — nothing executes until you iterate.

In [ ]:
def countdown(n: int):
    print("Starting countdown...")
    while n > 0:
        yield n  # This make our function to be generator
        n -= 1
    print("Finished...")

gen = countdown(5)

In [3]:
print(type(gen))

for num in gen:
    print(f" - {num}")

<class 'generator'>
Starting countdown...
 - 5
 - 4
 - 3
 - 2
 - 1
Finished...


## 5.2 `next()` — Manual Iteration

You can step through a generator manually with `next()`.

In [8]:
def simple_gen():
    yield "first"
    yield "second"

g = simple_gen()

print(next(g))

first


In [9]:
print(next(g))
try:
    print(next(g))
except Exception as e:
    print(f"Error: Stop Iteration")

second
Error: Stop Iteration


## 5.3 Memory Efficiency — Lists vs Generators

This is the **main reason** generators exist: processing large datasets with **O(1) memory**.

In [12]:
def numbers_list(n: int) -> list[int]:
    """Returns all numbers — O(n) memory."""
    return [i for i in range(n)]
 
def numbers_gen(n: int) -> Generator[int, None, None]:
    """Yields numbers one by one — O(1) memory."""
    for i in range(n):
        yield i

# Compare memory usage
n = 100_000
list_size = sys.getsizeof(numbers_list(n))
gen_size = sys.getsizeof(numbers_gen(n))
 
print(f"List of {n:,} ints: {list_size:,} bytes ({list_size/1024:.1f} KB)")
print(f"Generator object:  {gen_size} bytes")
print(f"Ratio: list is {list_size/gen_size:.0f}x larger than the generator!")

List of 100,000 ints: 800,984 bytes (782.2 KB)
Generator object:  200 bytes
Ratio: list is 4005x larger than the generator!


## 5.4 Generator Expressions

Like list comprehensions, but with `()` instead of `[]` — lazy evaluation.

In [13]:
# List comprehension — all values computed immediately
squares_list = [x**2 for x in range(10)]
 
# Generator expression — values computed on demand
squares_gen = (x**2 for x in range(10))
 
print(f"List: {squares_list}")
print(f"Generator: {squares_gen}")     # Shows object, not values
print(f"Values: {list(squares_gen)}")    # Consume into list
 
# Generator expressions in function calls
total = sum(x**2 for x in range(1000))   # No extra list in memory!
print(f"\nSum of squares 0-999: {total}")
 
# Chained with conditions
even_squares = (x**2 for x in range(20) if x % 2 == 0)
print(f"Even squares: {list(even_squares)}")

List: [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
Generator: <generator object <genexpr> at 0x73ce0b7e3370>
Values: [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]

Sum of squares 0-999: 332833500
Even squares: [0, 4, 16, 36, 64, 100, 144, 196, 256, 324]


## 5.5 `yield from` — Delegating to Sub-generators

`yield from` delegates iteration to another iterable or generator.

In [14]:
def gen_123():
    yield 1
    yield 2
    yield 3
 
def gen_abc():
    yield "a"
    yield "b"
    yield "c"
 
# Without yield from
def combined_manual():
    for item in gen_123(): # equal with yield from gen_123() below
        yield item
    for item in gen_abc():
        yield item
 
# With yield from — cleaner!
def combined():
    yield from gen_123()
    yield from gen_abc()
 
print(list(combined()))   # [1, 2, 3, 'a', 'b', 'c']

[1, 2, 3, 'a', 'b', 'c']


<h4 style="color: yellow"> This below if want solve it with other way. List of any depht must simplify</h4>

In [15]:
# Also works with any iterable
def flatten(nested: list) -> Generator:
    """Recursively flatten nested lists."""
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)   # Recursive delegation
        else:
            yield item
 
nested = [1, [2, 3], [4, [5, 6]], 7]
print(list(flatten(nested)))   # [1, 2, 3, 4, 5, 6, 7]

[1, 2, 3, 4, 5, 6, 7]


## 5.6 Infinite Generators

Generators can produce **infinite sequences** — only computed as needed.

In [ ]:
def naturals(start: int = 1) -> Generator[int, None, None]:
    """Infinite sequence of natural numbers."""
    n = start
    while True:
        yield n
        n += 1
 
# Take first 10 numbers
first_10 = list(islice(naturals(), 10))
print(f"First 10: {first_10}")

# And again take first 10 numbers
first_10 = list(islice(naturals(), 10)) # When we use the islice the generator start from the begining. 
print(f"First 10: {first_10}")
 
# Fibonacci — infinite
def fibonacci() -> Generator[int, None, None]:
    """Infinite Fibonacci sequence."""
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b
 
first_15_fib = list(islice(fibonacci(), 15))
print(f"First 15 Fibonacci: {first_15_fib}")

First 10: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
First 10: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
First 15 Fibonacci: [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377]


## 5.7 `.send()` — Two-Way Communication

Generators aren't just output — you can **send values into** them.
The sent value becomes the result of the `yield` expression.

In [17]:
def running_average():
    """Generator that computes a running average.
    Send values in, get current average out."""
    total = 0
    count = 0
    average = None
   
    while True:
        value = yield average  # Receive value, send back average
        if value is not None:
            total += value
            count += 1
            average = total / count
 
# Create and prime the generator
avg = running_average()
next(avg)  # Prime it — must call next() first to reach the yield
 
print(f"Send 10: avg = {avg.send(10)}")
print(f"Send 20: avg = {avg.send(20)}")
print(f"Send 30: avg = {avg.send(30)}")
print(f"Send 15: avg = {avg.send(15)}")

Send 10: avg = 10.0
Send 20: avg = 15.0
Send 30: avg = 20.0
Send 15: avg = 18.75


In [18]:
# Practical: state machine with send()
def traffic_light():
    """State machine — send 'next' to change light."""
    states = ["🔴 RED", "🟢 GREEN", "🟡 ORANGE"]
    index = 0
    while True:
        command = yield states[index]
        if command == "next":
            index = (index + 1) % len(states)
 
light = traffic_light()
print(next(light))           # 🔴 RED (initial state)
print(light.send("next"))    # 🟢 GREEN
print(light.send("next"))    # 🟡 ORANGE
print(light.send("next"))    # 🔴 RED
print(light.send("stay"))    # 🔴 RED (no change)
 
print("\n💡 .send() is how @contextmanager and async/await work under the hood")

🔴 RED
🟢 GREEN
🟡 ORANGE
🔴 RED
🔴 RED

💡 .send() is how @contextmanager and async/await work under the hood


## 5.8 Practical: Pipeline Pattern

Chain generators for efficient data processing — like Unix pipes.

In [19]:
# Data pipeline with generators — each step is lazy
 
def read_data():
    """Step 1: Simulate reading lines from a file."""
    data = ["  Alice, 30  ", "  Bob, 25  ", "  ", "  Charlie, 35  ", ""]
    yield from data
 
def clean(lines):
    """Step 2: Strip whitespace, skip empty lines."""
    for line in lines:
        cleaned = line.strip()
        if cleaned:
            yield cleaned
 
def parse(lines):
    """Step 3: Parse 'name, age' into dicts."""
    for line in lines:
        name, age = line.split(", ")
        yield {"name": name, "age": int(age)}
 
def adults_only(records):
    """Step 4: Filter to age >= 30."""
    for record in records:
        if record["age"] >= 30:
            yield record
 
# Build the pipeline — nothing runs yet!
pipeline = adults_only(parse(clean(read_data())))
 
# Consume — runs all stages lazily
for record in pipeline:
    print(f"  {record}")

  {'name': 'Alice', 'age': 30}
  {'name': 'Charlie', 'age': 35}


## 📋 Takeaways

| # | Concept | Key Point |
|---|---------|----------|
| 1 | `yield` | Pauses function, returns value, resumes on next call |
| 2 | Lazy evaluation | Nothing runs until you iterate |
| 3 | O(1) memory | Only one value in memory at a time |
| 4 | Generator expression | `(x**2 for x in range(n))` — lazy list comp |
| 5 | `yield from` | Delegate to sub-generator or iterable |
| 6 | `next(gen)` | Manual iteration, raises `StopIteration` when done |
| 7 | `.send(value)` | Two-way: send values *into* generators |
| 8 | Single-use | Cannot restart — create a new generator |
| 9 | Pipeline pattern | Chain generators for memory-efficient processing |
| 10 | `itertools` | `islice`, `chain`, `takewhile`, `count`, `groupby` |
| 11 | `batch()` pattern | `islice` in a loop for chunked processing |
| 12 | Java comparison | `yield` ≈ `Iterator` — but without the class boilerplate |